# 任务权重加载：从静默丢参到严格校验

目标：复现“模型可启动，任务适配参数却没有进入模型”的故障；对照真实加载源码和已保存审计，验证修复后的参数契约。输入为两参数教学树、真实加载源码、历史审计与正式评估；输出为丢失键列表、预期异常和修复后记录。

历史根因是禁用任务适配分支后，默认加载逻辑先对参数树取交集，静默丢弃多余参数。真实严格加载审计确认保留20个适配张量；旧的0/20起始基线因此失效。

## 使用方式与证据层次

每节先复现一个小型、可解释的错误，再展开真实源码、定位机制，最后核对历史设备记录与修复后的守卫。人工构造的参数树、状态探针和动作块只用于验证机制，不替代真实模型或成功率。默认不启动训练、模型服务或完整仿真。

真实模型检查须先选择已有匹配环境，设置 `AMD_DIAGNOSTICS_LIVE=1`，指定本地源码、配置、权重和独立输出目录；不下载资源，也不改原始实验目录。视频为已有完整回合，播放器直接内嵌数据。

In [1]:
from __future__ import annotations
from pathlib import Path
import json, os, sys, hashlib, dataclasses, tempfile, copy, time
import numpy as np
from IPython.display import display, Video
TOPIC = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "code/amd_benchmarks").is_dir())
EVIDENCE = TOPIC / "notebooks/results"
SNAPSHOTS = TOPIC / "code/diagnostics_sources"
DIAG = EVIDENCE / "diagnostics"
RUN_LIVE_CHECKS = os.environ.get("AMD_DIAGNOSTICS_LIVE", "0") == "1"
for row in json.loads((DIAG / "source-manifest.json").read_text()):
    assert hashlib.sha256((TOPIC / row["file"]).read_bytes()).hexdigest() == row["sha256"]
print("源码与证据指纹通过；默认运行小型复现和历史记录复算，真实模型检查：", RUN_LIVE_CHECKS)


源码与证据指纹通过；默认运行小型复现和历史记录复算，真实模型检查： False


## 1. 故障复现：能加载不等于参数齐全

下面只用两个数值参数重现“按模型形状取交集”的行为。模型关闭任务适配分支后，任务增量被当作多余键删掉；主干仍可加载。此处参数是教学夹具，真实模型参数不在课程包中。

In [2]:
params = {"backbone.weight": np.array([1.0]), "expert.lora_A": np.array([0.5])}
expected_without_adapter = {"backbone.weight"}
legacy_loaded = {k: v for k, v in params.items() if k in expected_without_adapter}
dropped = sorted(set(params) - set(legacy_loaded))
assert dropped == ["expert.lora_A"]
print("旧路径加载键：", sorted(legacy_loaded), "静默丢弃键：", dropped)


旧路径加载键： ['backbone.weight'] 静默丢弃键： ['expert.lora_A']


## 2. 定位：真实上游加载入口

这里逐行保留实际加载方法。关键顺序是创建模型结构、取交集、再检查树形；若模型结构本身已关闭适配分支，后面的检查也看不到被删除的张量。下方源代码作为方法定义展示，默认不构造真实模型。

In [3]:
if False:  # Source reference; real loading uses the checked config below.
    def load(self, params: at.Params, *, remove_extra_params: bool = True) -> "BaseModel":
        """Create a model with the given parameters."""
        model = nnx.eval_shape(self.create, jax.random.key(0))
        graphdef, state = nnx.split(model)
        if remove_extra_params:
            params = ocp.transform_utils.intersect_trees(state.to_pure_dict(), params)
        at.check_pytree_equality(expected=state.to_pure_dict(), got=params, check_shapes=True, check_dtypes=False)
        state.replace_by_pure_dict(params)
        return nnx.merge(graphdef, state)


## 3. 修复：由权重元数据选择模型，禁止静默删除

下面是课程实际部署路径的完整配置检查函数。无任务适配参数时使用主干结构；有任务适配参数时保留相应分支，结构不匹配就报错。严格加载强制关闭“删除多余参数”。

In [4]:
def checkpoint_model_config(model, checkpoint):
    """Keep task adapters present in the checkpoint and reject silent parameter dropping."""
    metadata = json.loads((Path(checkpoint) / 'params/_METADATA').read_text())
    adapter_keys = [key for key in metadata['tree_metadata'] if 'lora_' in key]
    if not adapter_keys:
        model = dataclasses.replace(model,
            paligemma_variant=model.paligemma_variant.removesuffix('_lora'),
            action_expert_variant=model.action_expert_variant.removesuffix('_lora'))
    elif not all('lora' in value for value in [model.paligemma_variant, model.action_expert_variant]):
        raise ValueError('Checkpoint contains task adapters, but the model disables them.')

    class StrictCheckpointConfig(type(model)):
        def load(self, params, *, remove_extra_params=False):
            return super().load(params, remove_extra_params=False)

    checked = StrictCheckpointConfig(**{field.name: getattr(model, field.name)
                                       for field in dataclasses.fields(model) if field.init})
    return checked, {'adapter_tensor_count': len(adapter_keys),
                     'paligemma_variant': checked.paligemma_variant,
                     'action_expert_variant': checked.action_expert_variant,
                     'remove_extra_params': False, 'loader_contract': 'strict-adapters-v2'}


## 4. 在小参数树上验证真实修复函数

用临时元数据调用上面的实际函数，分别验证：正确适配模型保留全部参数；禁用适配分支时明确拒绝；无适配参数的主干权重仍能加载。临时文件退出后删除。

In [5]:
@dataclasses.dataclass(frozen=True)
class SmallConfig:
    paligemma_variant: str = "gemma_2b_lora"
    action_expert_variant: str = "gemma_300m_lora"
    def load(self, params, *, remove_extra_params=True):
        expected = {"backbone.weight"}
        if "lora" in self.action_expert_variant:
            expected.add("expert.lora_A")
        actual = {k: v for k, v in params.items() if k in expected} if remove_extra_params else params
        if set(actual) != expected:
            raise ValueError("Parameter tree mismatch")
        return actual

with tempfile.TemporaryDirectory() as temp:
    root = Path(temp); (root / "params").mkdir()
    (root / "params/_METADATA").write_text(json.dumps({"tree_metadata": {k: {} for k in params}}))
    checked, audit = checkpoint_model_config(SmallConfig(), root)
    assert set(checked.load(params)) == set(params)
    try:
        checkpoint_model_config(SmallConfig("gemma_2b", "gemma_300m"), root)
    except ValueError as exc:
        print("预期拒绝：", exc)
    else:
        raise AssertionError("Adapter-disabled config must fail")
    (root / "params/_METADATA").write_text(json.dumps({"tree_metadata": {"backbone.weight": {}}}))
    plain, plain_audit = checkpoint_model_config(SmallConfig(), root)
    assert set(plain.load({"backbone.weight": params["backbone.weight"]})) == {"backbone.weight"}
print("修复函数夹具通过：", audit, plain_audit)


预期拒绝： Checkpoint contains task adapters, but the model disables them.
修复函数夹具通过： {'adapter_tensor_count': 1, 'paligemma_variant': 'gemma_2b_lora', 'action_expert_variant': 'gemma_300m_lora', 'remove_extra_params': False, 'loader_contract': 'strict-adapters-v2'} {'adapter_tensor_count': 0, 'paligemma_variant': 'gemma_2b', 'action_expert_variant': 'gemma_300m', 'remove_extra_params': False, 'loader_contract': 'strict-adapters-v2'}


## 5. 核对真实设备验证，保留旧结果失效原因

夹具证明检查机制；下面的20张量和5/20、3/20来自此前真实设备。两组正式结果均使用严格加载和预热，不能把它们与旧加载缺陷下的基线混在一起。

In [6]:
initial=json.loads((DIAG / "initial_investigation.json").read_text())
print("首次调查缺陷定位：",initial["defect"])
print("首次调查对归一化的判断：",initial["normalization_check"])
print("首次调查中的待确认状态属于当时记录；下面展示后续完成的严格加载验证。")


首次调查缺陷定位： {'location': 'amd_benchmarks.dexjoco.serve', 'old_behavior': 'Removed _lora suffix for official task checkpoint; upstream model.load intersect_trees silently dropped task adapter tensors', 'checkpoint_adapter_tensors': 20, 'fix': 'Select model from checkpoint metadata, preserve task adapters, force remove_extra_params=False', 'runtime_confirmation': 'pending'}
首次调查对归一化的判断： {'official_state_dimensions': 23, 'official_action_dimensions': 22, 'new_same_dimensions': True, 'maximum_absolute_quantile_difference': 0.00317661, 'conclusion': 'No gross normalization dimension/scale mismatch found; runtime interface audit remains.'}
首次调查中的待确认状态属于当时记录；下面展示后续完成的严格加载验证。


In [7]:
load_audit = json.loads((EVIDENCE / "system/strict_latency_model_load_audit.json").read_text())
formal = json.loads((EVIDENCE / "dexjoco/formal_comparison.json").read_text())
assert load_audit["adapter_tensor_count"] == 20
assert load_audit["remove_extra_params"] is False
for side in ("baseline", "trained"):
    row = formal[side]
    assert row["loader_contract"] == "strict-adapters-v2"
    print(side, row["successes"], "/", row["episodes"], row["checkpoint_sha256"])
print(load_audit)


baseline 5 / 20 21640047e97354a64b658c00396bb2832b19f1c9775c2b7582ca4c9994f1f594
trained 3 / 20 7cc8487f7609b05a255c6a942ff4f0f468302bf2bd6a08fafd3d4c7870115761
{'adapter_tensor_count': 20, 'paligemma_variant': 'gemma_2b_lora', 'action_expert_variant': 'gemma_300m_lora', 'remove_extra_params': False, 'loader_contract': 'strict-adapters-v2'}


## 6. 可选：在匹配环境加载已有真实权重

设置 `DEXJOCO_OPENPI_SOURCE` 为已有源码中的 `openpi` 目录、`DEXJOCO_CHECKPOINT` 为已缓存权重、`DEXJOCO_DIAGNOSTIC_CONFIG` 为修改好路径的设备配置，`DEXJOCO_DIAG_RUN_ROOT` 为全新外置目录。下方完整函数沿用项目已有模型构造和归一化加载；模型／分词器缺缓存时直接停止。此阶段只加载，不训练或评估。

In [8]:
def load_existing_policy():
    """Load a cached checkpoint using the audited model contract in an isolated run root."""
    source = Path(os.environ["DEXJOCO_OPENPI_SOURCE"]).resolve()
    checkpoint = Path(os.environ["DEXJOCO_CHECKPOINT"]).resolve()
    config_path = Path(os.environ["DEXJOCO_DIAGNOSTIC_CONFIG"]).resolve()
    run_root = Path(os.environ["DEXJOCO_DIAG_RUN_ROOT"]).resolve()
    assert (source / "src/openpi").is_dir()
    assert (checkpoint / "params/_METADATA").is_file()
    c = json.loads(config_path.read_text())
    c["source_root"] = str(source.parent)
    c["run_root"] = str(run_root)
    run_root.mkdir(parents=True, exist_ok=False)
    previous_cwd, previous_path, previous_env = Path.cwd(), sys.path[:], dict(os.environ)
    old_download = sys.modules.get("openpi.shared.download")
    old_cached_asset = getattr(old_download, "maybe_download", None)
    download_module = None
    try:
        sys.path.insert(0, str(TOPIC / "code"))
        from amd_benchmarks.dexjoco import get_config
        cfg = get_config(c)
        from openpi.shared import download as download_module
        from openpi.policies import policy_config
        checked, audit = checkpoint_model_config(cfg.model, checkpoint)
        cfg = dataclasses.replace(cfg, model=checked)
        policy = policy_config.create_trained_policy(cfg, checkpoint)
        print(audit)
        return policy
    finally:
        download_module = sys.modules.get("openpi.shared.download", download_module)
        if download_module is not None:
            if old_cached_asset is not None:
                download_module.maybe_download = old_cached_asset
            else:
                import importlib
                importlib.reload(download_module)
        os.chdir(previous_cwd)
        sys.path[:] = previous_path
        os.environ.clear()
        os.environ.update(previous_env)


In [9]:
# Test isolation after an intentional loader failure without importing a real model.
import types
from unittest.mock import patch
with tempfile.TemporaryDirectory() as temp:
    root=Path(temp); source=root/"openpi";(source/"src/openpi").mkdir(parents=True)
    checkpoint=root/"checkpoint";(checkpoint/"params").mkdir(parents=True)
    (checkpoint/"params/_METADATA").write_text("{}")
    config_path=root/"config.json";config_path.write_text("{}")
    pkg=types.ModuleType("amd_benchmarks");pkg.__path__=[]
    module=types.ModuleType("amd_benchmarks.dexjoco")
    download=types.ModuleType("openpi.shared.download")
    original_download=lambda value:value
    download.maybe_download=original_download
    def intentional_failure(config):
        os.chdir(root)
        os.environ["DIAGNOSTIC_FIXTURE_MUTATION"]="changed"
        download.maybe_download=lambda value:None
        raise RuntimeError("intentional loader failure")
    module.get_config=intentional_failure
    variables={"DEXJOCO_OPENPI_SOURCE":str(source),"DEXJOCO_CHECKPOINT":str(checkpoint),
               "DEXJOCO_DIAGNOSTIC_CONFIG":str(config_path),"DEXJOCO_DIAG_RUN_ROOT":str(root/"new_run")}
    with patch.dict(sys.modules,{"amd_benchmarks":pkg,"amd_benchmarks.dexjoco":module,"openpi.shared.download":download}),patch.dict(os.environ,variables):
        before=(Path.cwd(),sys.path[:],dict(os.environ))
        try: load_existing_policy()
        except RuntimeError as exc: print("预期加载异常：",exc)
        else: raise AssertionError("Fixture must fail before model construction")
        assert before==(Path.cwd(),sys.path[:],dict(os.environ))
        assert download.maybe_download is original_download
print("加载失败后目录、导入路径、环境和缓存下载函数均复原；未构造真实模型。")


预期加载异常： intentional loader failure
加载失败后目录、导入路径、环境和缓存下载函数均复原；未构造真实模型。


In [10]:
if RUN_LIVE_CHECKS:
    live_policy = load_existing_policy()
else:
    print("真实权重加载关闭；夹具与历史设备审计已完成。")


真实权重加载关闭；夹具与历史设备审计已完成。


## 7. 严格加载后的官方异步评估回放：起始权重，种子100

此录像来自已有评估回合；模型、条件与种子按下方文件路径标注，不将画面作为单一修复因素的因果证明。

In [11]:
display(Video(str(EVIDENCE / 'dexjoco/videos/base_seed_100.mp4'), embed=True))


## 复核要点

记录权重指纹、模型变体、适配参数数量和严格加载开关。报错比静默裁剪更容易定位。修改权重路径或结构后重新生成审计，不能沿用旧文件。后续归一化、预热和时序问题分别进入下一篇，不用单个成功率推断所有模块正确。